# D05 · 取消和什么有关

这一册只做三件能说清的事，**不做因果推断**：

1. **影响面**：每个因素把取消率拉开多少个百分点、涉及多少单；
2. **时机**：取消发生在到达前多少天，决定还有没有干预窗口；
3. **情景**：在写明假设的前提下，某类订单少取消能省多少。

先说方法上的两条底线：

- "超额取消单"= Σ nᵢ × max(0, rᵢ − r̄)，即高于总体平均的那些组多出来的取消。
  不用"最高组减最低组"，因为那等于假设所有组都能降到最低组的水平——
  停车位需求组是 0%，照那个算法能"消灭全部取消"。
- "结构性 / 可干预"是**业务判断**，写在 `cancel.FACTORS` 里逐条给理由，
  不是数据算出来的。把业务判断伪装成统计结论，是这类分析最常见的错。

In [ ]:
%autoreload 2
%matplotlib inline

import sqlite3

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from hotel_insight import cancel, config, rate, viz
from hotel_insight.viz import FAINT, INK, MAIN, MAIN_DARK, MAIN_LIGHT, MAIN_MID, MUTED

viz.apply_theme()
pd.set_option("display.width", 240)
pd.set_option("display.unicode.east_asian_width", True)

con = sqlite3.connect(config.DB_PATH)
s = config.load_settings()
a = s["analysis"]
NB = "D05"
SOURCE = "数据源：hotel.db / dwd_bookings（85,585 条入住记录，2015-07 至 2017-08）"

d = pd.read_sql("SELECT * FROM dwd_bookings", con)
print(f"记录数 {len(d):,}；取消 {int(d['is_canceled'].sum()):,} 条（{d['is_canceled'].mean():.2%}）")
assert len(d) == 85585

## 1 十二个因素的影响面

每个因素按"最高组 − 最低组"的极差、以及"超额取消单"两个口径量一次。
极差说明"这个因素能拉开多大差距"，超额取消单说明"涉及多少单"——
两个都要看：极差大的因素可能只有几百单，等于没法用。

In [ ]:
# 因素扫描：性质、极差、超额取消单
scan = cancel.factor_scan(d, s)
scan.assign(**{"最高组取消率": (scan["最高组取消率"] * 100).round(1),
               "最低组取消率": (scan["最低组取消率"] * 100).round(1),
               "极差(pp)": scan["极差(pp)"].round(1),
               "超额取消单": scan["超额取消单"].round(0),
               "覆盖单量": scan["覆盖单量"]})[
    ["性质", "组数", "最高组", "最高组取消率", "最低组", "最低组取消率",
     "极差(pp)", "超额取消单", "覆盖单量", "判定理由"]]

In [ ]:
# ── 出图 D05_01：各因素的超额取消单 ──
# 为什么用超额取消单而不是极差：极差回答"能差多少"，超额取消单回答"涉及多少单"，
# 后者才是决策依据。停车位极差 30pp，但只覆盖 7,206 单；市场细分极差 27pp，覆盖 85,583 单。
# 规范落在哪几行：按超额单降序 / 颜色按性质（结构性灰、混合中蓝、可干预主蓝、需存疑红）
#               只在副标题里说明"性质是人定的"
# ---- 1 指标 ----
NATURE_COLOR = {"结构性": viz.GRAY, "混合": viz.MAIN_MID,
                "可干预": viz.MAIN, "需存疑": viz.RED}
ex = scan.sort_values("超额取消单", ascending=False)   # 大的在上：hbar 会把第一行画在顶部
title = f"{ex.index[0].split('（')[0]}影响最大（{ex['超额取消单'].iloc[0]:,.0f} 单），但酒店改不了"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"超额取消单 = Σ nᵢ×max(0, rᵢ−总体均值)，共 {len(ex)} 个因素；"
            f"颜色是业务判断（结构性/混合/可干预/需存疑），不是统计结果")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.8, 5.0))
viz.hbar(ax, [i.split("（")[0] for i in ex.index], ex["超额取消单"].to_numpy(),
         value_fmt="{:,.0f}", colors=[NATURE_COLOR[x] for x in ex["性质"]])
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
# 图例用手工代理：四个色块，说明颜色含义
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c, label=k) for k, c in NATURE_COLOR.items()],
          frameon=False, fontsize=9, loc="lower right")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D05_01_因素影响面.png")
# ---- 4 读回断言：条长等于超额取消单 ----
assert np.allclose([b.get_width() for b in ax.patches], ex["超额取消单"].to_numpy())
assert len(ax.patches) == len(scan) == 12
plt.show()

排名本身有信息量：

- **市场细分 3,952 单**（结构性）、**提前期 3,711 单**（混合）、**客源国 3,036 单**（结构性）——前三名里两个是酒店短期改不了的；
- **特殊要求 2,511 单**、**停车位 2,007 单**、**历史取消 675 单**（可干预）——这三个才是能动的，但加起来也不到市场细分一个因素；
- **押金类型 693 单**（需存疑）——极差 70pp 是全场最大，却排在倒数第二，因为它只覆盖 1,037 单。

这张表最重要的读法不是排名，而是**"影响大的改不了，能改的影响小"**。
D06 会按这个矛盾给出行动建议。

## 2 影响面与可干预程度的四象限

把上表的两个维度画在一起：横轴是超额取消单（可算），纵轴是业务打分的可干预程度（人定）。

In [ ]:
# ── 出图 D05_02：影响面 × 可干预程度 ──
# 为什么用象限图：决策问题不是"谁影响最大"，而是"先动哪一个"，
# 两个判据缺一不可；单个排序会让人直接去啃市场细分这种改不了的因素。
# 规范落在哪几行：纵轴按档位显示文字（结构性/需存疑/混合/可干预），不露出 0.15、0.9 这种人为分值
# ---- 1 指标 ----
score = {"结构性": 0.00, "需存疑": 0.33, "混合": 0.66, "可干预": 1.00}
xs = scan["超额取消单"].to_numpy()
ys = np.array([score[x] for x in scan["性质"]])
labels = [i.split("（")[0] for i in scan.index]
title = "影响最大的两个（细分、客源国）都改不了，能改的只影响中等"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("横轴为超额取消单（可算）；纵轴为可干预程度（业务打分，"
            "结构性 0 / 需存疑 0.33 / 混合 0.66 / 可干预 1）")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.4, 5.8))
viz.quadrant(ax, xs, ys, labels, x_name="超额取消单（单）", y_name="",
             x_mid=1800, y_mid=0.50,
             y_ticks=([0.00, 0.33, 0.66, 1.00], ["结构性", "需存疑", "混合", "可干预"]))
ax.xaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D05_02_影响面与可干预.png")
# ---- 4 读回断言：点数等于因素数，且纵坐标就是打分 ----
assert len(ax.collections) == 1
pts = ax.collections[0].get_offsets()
assert len(pts) == len(scan)
assert np.allclose(sorted(pts[:, 1]), sorted(ys))
plt.show()

右上角（影响大 + 可干预）**是空的**——这是这份数据给出的最重要的一条结构性事实：
没有一个"影响又大又好改"的因素。

右下角（影响大 + 难控）：市场细分、客源国、客群类型、提前期。它们是预测的入口，不是干预的入口。
左上角（影响小 + 可干预）：特殊要求、停车位、历史取消。这才是行动的入口。

## 3 取消发生在什么时候：干预窗口有多宽

提前期是"订得多早"，取消时机是"什么时候退的"。两个不是一回事，
而**只有后者决定还来不来得及干预**。

In [ ]:
# ── 出图 D05_03：取消时机分布 ──
# 为什么用条形 + 累计线：单看分布不知道"提前 30 天提醒能覆盖多少"，
# 累计线直接给出覆盖率；这是决策需要的那个数。
# 规范落在哪几行：时段是有序变量（按到达前远近排）/ 累计线单独配色并标注 / 只统计取消单 /
#               左轴刻度隐去（柱上已有数字，两套百分比刻度会互相误读）
# ---- 1 指标 ----
tim = cancel.cancellation_timing(d, a["cancel_lead_bins"], a["cancel_lead_labels"])
within7 = tim["占比"].iloc[0]
within30 = tim["累计占比"].iloc[a["cancel_lead_labels"].index("8–30 天")]
title = f"{within7:.0%} 的取消在到达前 7 天内，{within30:.0%} 在 30 天内"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = (f"仅统计 {int(d['is_canceled'].sum()):,} 条取消单；"
            f"另有 {tim['占比'].iloc[-1]:.0%} 发生在 90 天以前")
# ---- 2 画图 ----
fig, ax = plt.subplots(figsize=(9.4, 4.2))
xs_ = np.arange(len(tim))
ax.bar(xs_, tim["占比"].to_numpy() * 100, color=MAIN_MID, width=0.6, zorder=2)
for x, v in zip(xs_, tim["占比"].to_numpy()):
    ax.text(x, v * 100 + 0.8, f"{v:.0%}", ha="center", va="bottom", fontsize=9, color=INK)
ax.set_xticks(xs_, tim.index, fontsize=9)
ax.set_yticks([])                                  # 左轴隐去：柱上已有数，两套百分比刻度会误读
ax.spines["left"].set_visible(False)
ax.set_ylim(0, tim["占比"].max() * 135)
ax.grid(axis="x", visible=False)
ax2 = ax.twinx()
ax2.plot(xs_, tim["累计占比"].to_numpy() * 100, color=MAIN_DARK, marker="o",
         markersize=5, linewidth=1.6, zorder=3)
ax2.set_ylim(0, 112)
ax2.set_yticks([0, 25, 50, 75, 100], ["0", "25%", "50%", "75%", "100%"])
ax2.set_ylabel("累计占比", color=MAIN_DARK, fontsize=9.5)
ax2.tick_params(axis="y", labelsize=8.5, colors=MAIN_DARK)
for x, v in zip(xs_, tim["累计占比"].to_numpy()):
    ax2.annotate(f"{v:.0%}", (x, v * 100), textcoords="offset points",
                 xytext=(0, 9), ha="center", fontsize=8.5, color=MAIN_DARK)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D05_03_取消时机.png")
# ---- 4 读回断言：累计占比末位为 1 ----
assert abs(tim["累计占比"].iloc[-1] - 1) < 1e-12
assert np.allclose([b.get_height() for b in ax.patches], tim["占比"].to_numpy() * 100)
plt.show()

一个反直觉的结果：**46% 的取消发生在到达前 30 天以上**（累计到 30 天只有 46%），
即一半以上的取消是"很早以前就退了"。

这意味着：

- 在 T−7 天发提醒，最多只覆盖 **19%** 的取消——晚了；
- 在 T−30 天做再确认，能覆盖 **46%**；
- 剩下 54% 发生在 30 天以前，要在**预订后一周内**就介入。

这条结论直接决定 D06 的情景参数选 90 天而不是 7 天。

## 4 取消涉及的盘子有多大

先说口径：`adr × total_nights` 是**名义收入**，不是实际损失。取消后房间多半会被卖掉一部分，
真实损失要用每日房态与替代率算，本数据没有。

In [ ]:
rev = cancel.revenue_at_risk(d)
print(f"取消单数        {rev['取消单数']:>10,} 条")
print(f"取消单名义收入  {rev['取消订单名义收入']:>14,.0f}")
print(f"全部名义收入    {rev['全部名义收入']:>14,.0f}")
print(f"取消占比        {rev['占比']:>14.1%}")
print(f"每取消单均值    {rev['每单均值']:>14,.0f}")

# ---- 对账：取消单的名义收入 + 未取消单的名义收入 = 全部 ----
rev_all = d["adr"] * d["total_nights"]
assert abs(rev_all[d["is_canceled"] == 1].sum() - rev["取消订单名义收入"]) < 1e-6
assert abs(rev_all.sum() - rev["全部名义收入"]) < 1e-6
print("✅ 对账通过：两部分收入相加等于全部名义收入")

取消单占记录数的 27.9%，却占名义收入的 **33.3%**——因为取消的订单平均更贵
（每单 482，高于全体的 402）。高房价订单更容易取消，这与 D04 的"价格与取消率同向"一致。

**这不是"损失 1,148 万"**。正确的说法是"取消涉及的盘子是 1,148 万名义收入"，
真实损失远小于此。

## 5 三个看起来像结论、其实是口径的陷阱

因素扫描里有三个数字漂亮得不像真的：
押金类型极差 70pp、停车位需求 30pp、历史取消 1 次 76.75%。
三个都不该直接当结论用。

In [ ]:
# ── 出图 D05_04：三个陷阱 ──
# 为什么画在一张图上：三个陷阱的性质相同（都是字段口径问题，不是业务发现），
# 并排能让人一次记住"这些数不能直接用"。
# 规范落在哪几行：小多图共享一套配色 / 每组标样本量（陷阱往往是"样本极小"造成的）/ 一行结论在副标题
# ---- 1 指标 ----
dep = rate.rate_table(d, "deposit_type")
park = rate.rate_table(d, "required_car_parking_spaces", sort="分组", ascending=True).head(2)
hist = rate.rate_table(d, "previous_cancellations", sort="分组", ascending=True).head(2)
nr = dep.loc["不可退款"]
title = f"不可退款订单取消 {nr['取消率']:.0%}，是'无押金'的 {nr['取消率'] / dep.loc['无押金', '取消率']:.1f} 倍"
assert len(title) <= 40, f"标题 {len(title)} 字，超过 40 字"
subtitle = ("三个都不像真实业务：押金 94.7%、有停车位 0%、历史取消 1 次 76.8%；"
            "样本量见括号，先查口径再当结论")
# ---- 2 画图 ----
fig, axes = plt.subplots(1, 3, figsize=(12.4, 3.4))
panels = [
    (dep, "押金类型", lambda x: f"{x}（{int(dep.loc[x, '记录数']):,}）"),
    (park, "停车位需求", lambda x: f"{x} 个（{int(park.loc[x, '记录数']):,}）"),
    (hist, "历史取消次数", lambda x: f"{x} 次（{int(hist.loc[x, '记录数']):,}）"),
]
for ax, (t, name, lab) in zip(axes, panels):
    viz.hbar(ax, [lab(i) for i in t.index], t["取消率"].to_numpy(), value_fmt="{:.0%}",
             colors=[viz.RED if v > 0.9 or v == 0 else viz.MAIN_MID for v in t["取消率"]])
    ax.xaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
    ax.set_xlim(0, 1.15)
    ax.set_title(name, fontsize=10, color=MUTED, loc="left")
    ax.tick_params(labelsize=8.5)
# ---- 3 收尾 ----
viz.finish(fig, title, subtitle, SOURCE, "D05_04_三个口径陷阱.png")
# ---- 4 读回断言 ----
assert np.allclose([b.get_width() for b in axes[0].patches], dep["取消率"].to_numpy())
assert (park.loc[park.index > 0, "取消率"] == 0).all(), "有停车位的取消率应为 0"
assert f"{nr['取消率']:.0%}" in fig.texts[0].get_text()
plt.show()

三个陷阱的成因不一样，处理方式也不同：

**1. 押金类型（不可退款 94.7%）**
"不可退款"在系统里记录的往往是**已经取消且不退款的订单**，而不是"预订时选择了不可退款"。
这是从字段值反推业务含义的经典错误。
【仅为推测，无实证】要确认得看订单状态流转日志，本数据没有。
处理：不放进杠杆清单，标记为"需存疑"。

**2. 停车位需求（有需求 0% 取消）**
7,206 条"有停车位需求"的订单，取消率恰好是 0。真实业务里不可能有这么干净的 0。
更可能的解释是：这个字段只在**入住登记**时才写入，所以天然只存在于到店订单。
处理：不能反过来推"承诺车位能消灭取消"，那是循环论证。

**3. 历史取消次数（1 次 76.8%，2 次以上反而降）**
如果"历史取消"真是风控信号，2 次、3 次应该更高，实际却掉到 34%、28%。
说明这个字段统计的可能是"本次之前的取消次数"，而高频取消客户在这份数据里很少
（1 次 1,368 条、2 次 93 条、3 次 54 条），小样本把曲线拉成了锯齿。
处理：可以用于风控（1 次这一档有 1,368 条、取消率 76.8%，区间很窄），但不能解释成"越多越危险"。

## 6 情景演算：能省多少，取决于你信哪个假设

对提前期 > 90 天的订单做一次再确认，假设该组取消率**相对下降 10%**，能少取消多少单？

**这是情景，不是预测。** 10% 是 `settings` 里写死的假设值，没有任何一版数据能验证它。

In [ ]:
sc = cancel.scenario_reconfirm(d, s)
print(f"适用单量        {sc['适用单量']:>8,} 条（占 {sc['适用单量占比']:.1%}）")
print(f"当前取消率      {sc['当前取消率']:>8.1%}")
print(f"少取消单数      {sc['少取消单数']:>8,.0f} 条")
print(f"少取消名义收入  {sc['少取消名义收入']:>8,.0f}")
print(f"假设            {sc['假设']}")

# 敏感性：假设从 5% 到 30%，结果线性放大——因为公式是"单数×取消率×drop"，
# 唯一的非线性来自"哪些单适用"，而适用集不随 drop 变
rows = []
for drop in (0.05, 0.10, 0.20, 0.30):
    s2 = {**s, "analysis": {**a, "scenario_relative_drop": drop}}
    r = cancel.scenario_reconfirm(d, s2)
    rows.append({"相对下降": f"{drop:.0%}", "少取消单数": round(r["少取消单数"]),
                 "少取消名义收入": round(r["少取消名义收入"])})
pd.DataFrame(rows)

三条必须一起说清的话，缺一条这个数就会被误用：

1. **适用集是"提前期 > 90 天的订单"**，共 29,753 条，占 34.8%——这批单下单时就能识别；
2. **10% 是假设**，不是实测。真实效果只有做 A/B 才知道，而本数据没有实验；
3. **少取消的"名义收入"不是净收益**：再确认本身有成本（人力、渠道费、客户流失），本数据算不了。

## 7 本册结论

| 问题 | 答案 | 可信度 |
| --- | --- | --- |
| 取消和什么有关 | 市场细分（3,952 单）> 提前期（3,711）> 客源国（3,036）> 特殊要求（2,511） | 高（样本充分） |
| 哪些能改 | 特殊要求、停车位、历史取消风控 | 中（性质是业务判断） |
| 什么时候退 | 19% 在 T−7 内，46% 在 T−30 内，54% 更早 | 高（口径清晰） |
| 盘子多大 | 名义收入 1,148 万，占 33.3% | 高（但非损失） |
| 能省多少 | 假设降 10%，少 1,102 单 / 57.7 万名义收入 | **低（假设驱动）** |

**不能答的**：为什么退（没有取消原因字段）、退了之后有没有再订（没有客户 ID 串联）、
真实损失多少（没有每日房态）。

## 8 发现清单（本册）

| # | 发现 | 影响 |
| --- | --- | --- |
| 1 | 右上象限（影响大+可干预）为空 | 没有"又好又大"的杠杆，行动只能从小处开始 |
| 2 | 54% 的取消发生在 T−30 以前 | T−7 提醒太晚，干预要前置到预订后一周 |
| 3 | 押金"不可退款"94.7% | 字段值可能是取消后的状态，不是预订时的选择 |
| 4 | 有停车位需求 0% 取消（7,206 条） | 字段可能只在入住时写入，存在循环论证风险 |
| 5 | 历史取消 1 次 76.8%、2 次以上反而降 | 小样本锯齿，只能做风控特征，不能解释成梯度 |
| 6 | 取消单名义收入占比 33.3% > 记录数占比 27.9% | 高房价订单更易取消 |

In [ ]:
con.close()
print("D05 完成：影响面、时机、情景三件事已量化，三个口径陷阱已标注")